In [9]:
# Change your team name to Group_xx
TEAM_NAME = 'Group_02'

In [10]:
# NOTE: you CAN change this cell
# If you want to use your own database, download it here
# !gdown ...

In [11]:
# NOTE: you CAN change this cell
# Add more to your needs
# you must place ALL pip install here
!pip install editdistance
!pip install daachorse

In [12]:
# NOTE: you CAN change this cell
# import your library here
import time
import daachorse

In [13]:
# NOTE: you MUST change this cell
# New methods / functions must be written under class Solution.
class Solution:
    def __init__(
        self,
        list_province: list[str],
        list_ward: list[str],
        list_street: list[str]
    ):
        self.list_province = list_province
        self.list_ward = list_ward
        self.list_street = list_street

        self.entity_map = {}

        def add_entities(entities: list[str], entity_type: str):
            for original_name in entities:
                name_lower = " ".join(original_name.lower().split())
                if not name_lower: continue
                if name_lower not in self.entity_map:
                    self.entity_map[name_lower] = {}
                self.entity_map[name_lower][entity_type] = original_name

        add_entities(self.list_province, "province")
        add_entities(self.list_ward, "ward")
        add_entities(self.list_street, "street")

        self.patterns = list(self.entity_map.keys())

        self.automaton = daachorse.CharwiseDoubleArrayAhoCorasick(
            self.patterns,
            match_kind=daachorse.MATCH_KIND_LEFTMOST_LONGEST
        )

    def process(self, s: str):
        s_lower = " ".join(s.lower().split())
        valid_matches = []

        for match in self.automaton.find(s_lower):
            start = match[0]
            end = match[1]
            pattern_id = match[2]
            pat_str = self.patterns[pattern_id]

            left_ok, right_ok = True, True
            if start > 0:
                lc = s_lower[start-1]
                if lc.isdigit() and pat_str[0].isdigit():
                    left_ok = False
            if end < len(s_lower):
                rc = s_lower[end]
                if rc.isdigit() and pat_str[-1].isdigit():
                    right_ok = False

            if left_ok and right_ok:
                pat_dict = self.entity_map[pat_str]
                valid_matches.append((start, end, pat_dict))

        valid_matches.sort(key=lambda x: x[0], reverse=True)

        result = {
            "province": None,
            "ward": None,
            "street": None,
        }

        used_intervals = []
        def is_overlap(st, en):
            for us, ue in used_intervals:
                if not (en <= us or st >= ue): return True
            return False

        for entity_type in ["province", "ward", "street"]:
            for st, en, pat_dict in valid_matches:
                if entity_type in pat_dict and not is_overlap(st, en):
                    result[entity_type] = pat_dict[entity_type]
                    used_intervals.append((st, en))
                    break

        return result

In [14]:
# Download public test
!rm -rf test.json
!gdown --fuzzy https://drive.google.com/file/d/18qzaHmx9-fcKPi4gTq304G2JvXy_g6XW/view?usp=sharing -O test.json
!gdown --fuzzy https://drive.google.com/file/d/1PHRkRLRaO2vFivLbcPJsaxjvZ7K_I3lU/view?usp=drive_link -O list_province.txt
!gdown --fuzzy https://drive.google.com/file/d/1QrSouYn6IUEW7Impje1vOxzsksjlAJ-h/view?usp=drive_link -O list_ward.txt
!gdown --fuzzy https://drive.google.com/file/d/1yP2dBE8K9QtUiaLPewtQRr-hDH0gOgfa/view?usp=drive_link -O list_street.txt

Downloading...
From: https://drive.google.com/uc?id=18qzaHmx9-fcKPi4gTq304G2JvXy_g6XW
To: /content/test.json
100% 42.7k/42.7k [00:00<00:00, 14.6MB/s]
Downloading...
From: https://drive.google.com/uc?id=1PHRkRLRaO2vFivLbcPJsaxjvZ7K_I3lU
To: /content/list_province.txt
100% 392/392 [00:00<00:00, 1.15MB/s]
Downloading...
From: https://drive.google.com/uc?id=1QrSouYn6IUEW7Impje1vOxzsksjlAJ-h
To: /content/list_ward.txt
100% 1.65k/1.65k [00:00<00:00, 5.74MB/s]
Downloading...
From: https://drive.google.com/uc?id=1yP2dBE8K9QtUiaLPewtQRr-hDH0gOgfa
To: /content/list_street.txt
100% 6.65k/6.65k [00:00<00:00, 19.1MB/s]


In [15]:
def _judge_normalize(s: str, context=None):
    return s

EXCEL_FILE = f'{TEAM_NAME}.xlsx'

import json
import time
with open('test.json') as f:
    data = json.load(f)

with open("list_province.txt") as f:
    list_province = f.read().strip().splitlines()

with open("list_ward.txt") as f:
    list_ward = f.read().strip().splitlines()

with open("list_street.txt") as f:
    list_street = f.read().strip().splitlines()

summary_only = True
df = []
solution = Solution(
    list_province=list_province,
    list_ward=list_ward,
    list_street=list_street,
)
timer = []
correct = 0
for test_idx, data_point in enumerate(data):
    address = data_point["address"]

    ok = 0
    try:
        answer = data_point["result"]
        answer["province_normalized"] = _judge_normalize(answer["province"])
        answer["ward_normalized"] = _judge_normalize(answer["ward"])
        answer["street_normalized"] = _judge_normalize(answer["street"])

        start = time.perf_counter_ns()
        result = solution.process(address)
        finish = time.perf_counter_ns()
        timer.append(finish - start)
        result["province_normalized"] = _judge_normalize(result["province"])
        result["ward_normalized"] = _judge_normalize(result["ward"])
        result["street_normalized"] = _judge_normalize(result["street"])

        province_correct = int(answer["province_normalized"] == result["province_normalized"])
        ward_correct = int(answer["ward_normalized"] == result["ward_normalized"])
        street_correct = int(answer["street_normalized"] == result["street_normalized"])
        ok = province_correct + street_correct + ward_correct

        df.append([
            test_idx,
            address,

            answer["province"],
            result["province"],
            answer["province_normalized"],
            result["province_normalized"],
            province_correct,

            answer["ward"],
            result["ward"],
            answer["ward_normalized"],
            result["ward_normalized"],
            ward_correct,

            answer["street"],
            result["street"],
            answer["street_normalized"],
            result["street_normalized"],
            street_correct,

            ok,
            timer[-1] / 1_000_000_000,
        ])
    except Exception as e:
        print(f"{answer = }")
        print(f"{result = }")
        df.append([
            test_idx,
            address,

            answer["province"],
            "EXCEPTION",
            answer["province_normalized"],
            "EXCEPTION",
            0,

            answer["ward"],
            "EXCEPTION",
            answer["ward_normalized"],
            "EXCEPTION",
            0,

            answer["street"],
            "EXCEPTION",
            answer["street_normalized"],
            "EXCEPTION",
            0,

            0,
            0,
        ])
        # any failure count as a zero correct
        pass
    correct += ok


    if not summary_only:
        # responsive stuff
        print(f"Test {test_idx:5d}/{len(data):5d}")
        print(f"Correct: {ok}/3")
        print(f"Time Executed: {timer[-1] / 1_000_000_000:.4f}")


print(f"-"*30)
total = len(data) * 3
score_scale_10 = round(correct / total * 10, 2)
if len(timer) == 0:
    timer = [0]
max_time_sec = round(max(timer) / 1_000_000_000, 4)
avg_time_sec = round((sum(timer) / len(timer)) / 1_000_000_000, 4)

import pandas as pd

df2 = pd.DataFrame(
    [[correct, total, score_scale_10, max_time_sec, avg_time_sec]],
    columns=['correct', 'total', 'score / 10', 'max_time_sec', 'avg_time_sec',],
)

columns = [
    'ID',
    'text',

    'province',
    'province_student',
    'province_normalized',
    'province_student_normalized',
    'province_correct',

    'ward',
    'ward_student',
    'ward_normalized',
    'ward_student_normalized',
    'ward_correct',

    'street',
    'street_student',
    'street_normalized',
    'street_student_normalized',
    'street_correct',

    'total_correct',
    'time_sec',
]

df = pd.DataFrame(df)
df.columns = columns

print(f'{TEAM_NAME = }')
print(f'{EXCEL_FILE = }')
print(df2)

!pip install xlsxwriter
writer = pd.ExcelWriter(EXCEL_FILE, engine='xlsxwriter')
df2.to_excel(writer, index=False, sheet_name='summary')
df.to_excel(writer, index=False, sheet_name='details')
writer.close()

------------------------------
TEAM_NAME = 'Group_02'
EXCEL_FILE = 'Group_02.xlsx'
   correct  total  score / 10  max_time_sec  avg_time_sec
0      595    600        9.92           0.0           0.0
